# 02 — Integration (silver layer)

**Purpose.** Phase 2 silver-layer integration: apply the
PROVENANCE-deferred preprocessing the bronze layer left untouched,
canonicalise join keys across sources, and produce the two analytical
tables that feed Phase 4 visual components.

**Inputs.** The eleven Phase 1 bronze parquets under
`../data/processed/{epoch,ember,im3,aqueduct,jegham}/`, plus the AWS
raw CSV at `../data/raw/jegham/AWS_Env_Multipliers.csv` (re-read with
`skiprows=1` per PROVENANCE).

**Outputs.**

- `../data/processed/integrated/models_enriched.parquet`
- `../data/processed/integrated/regions_geographic.parquet`
- `../data/processed/integrated/README.md`

**Phase 2 design decision.** Jegham's pre-computed per-query energy /
water / carbon values remain the spine for per-query reporting
(Component 1). The `regions_geographic` table provides geographic
*context* (grid carbon intensity, water stress, US facility footprint)
and does **not** override Jegham's numbers by recombining
`Jegham energy × multiplier CIF/WUE`.

## A. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
INTEGRATED = PROCESSED / "integrated"
INTEGRATED.mkdir(parents=True, exist_ok=True)


# Adapted from notebooks/01_ingestion.ipynb. Copied rather than imported
# because notebooks should not import from each other.
def validate_and_write(df, filename, must_have_columns, expected_min_rows):
    missing = [c for c in must_have_columns if c not in df.columns]
    assert not missing, f"{filename}: missing required columns {missing}"
    assert (
        len(df) >= expected_min_rows
    ), f"{filename}: {len(df)} rows < expected min {expected_min_rows}"

    print(f"{filename}: rows={len(df)} cols={len(df.columns)}")

    out = INTEGRATED / f"{filename}.parquet"
    df.to_parquet(out, engine="pyarrow", index=False)
    return out

## B. PROVENANCE-deferred preprocessing

This section re-reads or filters bronze parquets to apply the
preprocessing the bronze layer deliberately deferred. Each subsection
corresponds to one source file and applies one operation documented in
that source's `PROVENANCE.md`.

### B.1 AWS multipliers — re-read with `skiprows=1`

The bronze parquet was written without `skiprows` to keep ingestion
faithful; AWS PROVENANCE documents that the real headers sit on row 1
behind a banner row. Re-read from RAW with `skiprows=1`.

In [2]:
df_aws = pd.read_csv(RAW / "jegham" / "AWS_Env_Multipliers.csv", skiprows=1)
print(f"df_aws shape: {df_aws.shape}")
print(f"df_aws columns: {list(df_aws.columns)}")

df_aws shape: (11, 7)
df_aws columns: ['Unnamed: 0', 'Region', 'WUE Source', 'CIF', 'Multiplier', 'WUE Source.1', 'CIF.1']


### B.2 Microsoft multipliers — bronze parquet directly

Microsoft PROVENANCE does **not** document a banner row; the Phase 2
scoping pass confirmed `skiprows=1` would consume the first data row
as a header. Read the bronze parquet directly.

In [3]:
df_ms = pd.read_parquet(
    PROCESSED / "jegham" / "Microsoft_Env_Multipliers.parquet"
)
print(f"df_ms shape: {df_ms.shape}")
print(f"df_ms columns: {list(df_ms.columns)}")

df_ms shape: (28, 4)
df_ms columns: ['Unnamed: 0', 'Region', 'WUE Source', 'CIF']


### B.3 Ember yearly — filter to `Variable == 'CO2 intensity'`

Ember's long-format file carries dozens of `Variable`s (generation,
demand, emissions, intensity, etc.). The integration only consumes
grid carbon intensity.

In [4]:
ember_yr_full = pd.read_parquet(PROCESSED / "ember" / "ember_yearly.parquet")
df_ember_yr = ember_yr_full[
    ember_yr_full["Variable"] == "CO2 intensity"
].copy()
print(f"df_ember_yr shape after filter: {df_ember_yr.shape}")
assert {"Area", "ISO 3 code", "Year", "Value"}.issubset(df_ember_yr.columns)

df_ember_yr shape after filter: (5751, 18)


### B.4 Ember US-states — filter to `CO2 intensity` and drop the `'US'` national-aggregate row

Ember-US tags the national total with `State code == 'US'` (state
name `'US Total'`). For the state-level join we keep only true states
+ DC + PR.

In [5]:
ember_us_full = pd.read_parquet(PROCESSED / "ember" / "ember_us_states.parquet")
ember_us_co2 = ember_us_full[
    ember_us_full["Variable"] == "CO2 intensity"
].copy()
print(f"after CO2 intensity filter: {ember_us_co2.shape}")
df_ember_us = ember_us_co2[ember_us_co2["State code"] != "US"].copy()
print(f"after dropping 'US' aggregate: {df_ember_us.shape}")

after CO2 intensity filter: (1310, 13)
after dropping 'US' aggregate: (1285, 13)


### B.5 IM3 atlas — filter to `type == 'campus'`

The atlas mixes multiple geographic layers (`building`, `campus`,
`point`); the implementation plan specifies the `campus` layer for
Component 3. Operator-name backfill from the `name` field is
documented in the project plan but is **not** required for the
state-aggregate grain we produce — we count facilities and sum sqft
by state, not by operator. Skipping that backfill here.

In [6]:
im3_full = pd.read_parquet(
    PROCESSED / "im3" / "im3_open_source_data_center_atlas_v2026_02_09.parquet"
)
df_im3 = im3_full[im3_full["type"] == "campus"].copy()
print(f"df_im3 shape after type=='campus' filter: {df_im3.shape}")

df_im3 shape after type=='campus' filter: (135, 13)


### B.6 Aqueduct country — filter to `bws` + `Tot` and replace `-9999` sentinels

Aqueduct ships several indicators (`bws`, `rfr`, `drr`) and weighting
schemes (`Dom`, `Ind`, `Ag`, `Tot`). The integration uses baseline
water stress (`bws`) at total weighting (`Tot`). Singapore's `bws` row
is sentinel-coded `-9999`; replace with NaN.

In [7]:
aq_country_full = pd.read_parquet(
    PROCESSED / "aqueduct" / "Aqueduct40_country_baseline.parquet"
)
df_aq_country = aq_country_full[
    (aq_country_full["indicator_name"] == "bws")
    & (aq_country_full["weight"] == "Tot")
].copy()
df_aq_country = df_aq_country.replace(-9999, np.nan)
print(f"df_aq_country shape after bws+Tot filter: {df_aq_country.shape}")

df_aq_country shape after bws+Tot filter: (189, 10)


### B.7 Aqueduct province — US only, `bws` + `Tot`, sentinel handling

Same indicator/weight filter as B.6, plus restriction to US provinces
(`gid_0 == 'USA'`).

In [8]:
aq_prov_full = pd.read_parquet(
    PROCESSED / "aqueduct" / "Aqueduct40_province_baseline.parquet"
)
df_aq_us = aq_prov_full[
    (aq_prov_full["gid_0"] == "USA")
    & (aq_prov_full["indicator_name"] == "bws")
    & (aq_prov_full["weight"] == "Tot")
].copy()
df_aq_us = df_aq_us.replace(-9999, np.nan)
print(f"df_aq_us shape after USA + bws + Tot filter: {df_aq_us.shape}")

df_aq_us shape after USA + bws + Tot filter: (51, 12)


## C. Key canonicalisation

Canonicalise join keys to match across sources.

- **Country.** ISO-3 alpha codes are native on both sides: Ember
  `'ISO 3 code'` ↔ Aqueduct `'gid_0'`. No mapping needed.
- **US state.** Two-letter postal codes are native on Ember-US
  (`'State code'`) and IM3 (`'state_abb'`). Aqueduct-province only
  publishes the full state name (`'name_1'`), so we apply a 51-row
  inline mapping (50 states + DC) here.

In [9]:
us_state_to_postal = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "District of Columbia": "DC",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
}

df_aq_us["state_abb"] = df_aq_us["name_1"].map(us_state_to_postal)
mapped = df_aq_us["state_abb"].notna().sum()
missing_names = sorted(
    df_aq_us.loc[df_aq_us["state_abb"].isna(), "name_1"].dropna().unique().tolist()
)
print(f"Aqueduct US mapping: {mapped} of {len(df_aq_us)} rows mapped")
if missing_names:
    print(f"unmapped name_1 values: {missing_names}")
assert not missing_names, (
    f"Aqueduct US name_1 values not in postal map: {missing_names}"
)

Aqueduct US mapping: 51 of 51 rows mapped


## D. Build `models_enriched.parquet` (Jegham × Epoch)

Per-(model, query length) table. Spine: Jegham `DataSnapshotOct26`
(194 rows). Enriched with Epoch training metadata where the model name
matches.

**Match policy.** Case-insensitive after stripping whitespace from
Epoch's `Model` (Epoch carries leading-space and TeX-escape artifacts)
and stripping parenthetical disambiguators from Jegham's `Model`
(e.g. `'GPT-4o (Aug)'` → `'GPT-4o'`, `'DeepSeek R1 (Azure)'` →
`'DeepSeek R1'`).

**Join.** Left-outer on `match_key`. All 194 Jegham rows preserved;
Epoch columns are NaN where unmatched.

In [10]:
df_jegham = pd.read_parquet(
    PROCESSED / "jegham" / "DataSnapshotOct26.parquet"
)
assert "Model" in df_jegham.columns
print(f"df_jegham: {df_jegham.shape}, unique models: {df_jegham['Model'].nunique()}")

df_jegham: (194, 98), unique models: 66


In [11]:
df_epoch = pd.read_parquet(PROCESSED / "epoch" / "epoch_all_models.parquet")
assert {
    "Model",
    "Publication date",
    "Organization",
    "Training compute (FLOP)",
    "Parameters",
    "Training hardware",
    "Training power draw (W)",
}.issubset(df_epoch.columns)
print(f"df_epoch: {df_epoch.shape}")

df_epoch: (3509, 57)


In [12]:
# Match keys are derived columns used only for the join; they are
# dropped from the final result. The regex removes '(...)' segments
# along with their surrounding whitespace.
df_jegham["match_key"] = (
    df_jegham["Model"]
    .str.replace(r"\s*\(.*?\)\s*", "", regex=True)
    .str.strip()
    .str.lower()
)
df_epoch["match_key"] = df_epoch["Model"].str.strip().str.lower()

In [13]:
epoch_keys = set(df_epoch["match_key"].dropna())
jegham_unique = (
    df_jegham[["Model", "match_key"]].drop_duplicates("Model").reset_index(drop=True)
)
matched_mask = jegham_unique["match_key"].isin(epoch_keys)
n_matched = int(matched_mask.sum())
n_total = len(jegham_unique)
print(f"Jegham unique models: {n_total}")
print(f"Matched (case-insensitive, paren-stripped) to Epoch: {n_matched}")
print(f"Unmatched: {n_total - n_matched}")
unmatched_models = sorted(jegham_unique.loc[~matched_mask, "Model"].tolist())
print("\nUnmatched Jegham models (Phase 2 disposition: Epoch metadata absent):")
for m in unmatched_models:
    print(f"  {m!r}")

Jegham unique models: 66
Matched (case-insensitive, paren-stripped) to Epoch: 45
Unmatched: 21

Unmatched Jegham models (Phase 2 disposition: Epoch metadata absent):
  'Claude 4 Opus'
  'Claude 4 Sonnet'
  'DeepSeek R1 (Azure)'
  'DeepSeek R1 (DeepSeek) [128k]'
  'DeepSeek V3 (Azure)'
  'DeepSeek V3 (DeepSeek)'
  'GPT-4'
  'GPT-4 Turbo'
  'Gemini 2.5 Pro (AI Studio)'
  'Gemini 2.5 Pro Vertex'
  'Grok 3 Fast'
  'Grok 3 mini Reasoning (high)'
  'Grok 3 mini Reasoning (high) Fast'
  'Llama 3 70B'
  'Llama 3 8B'
  'Llama 3.1 405B Latency Optimized'
  'Llama 3.1 405B Standard'
  'Llama 3.1 70B Latency Optimized'
  'Llama 3.1 70B Standard'
  'Llama 3.1 8B'
  'Mistral Small (Feb)'


In [14]:
# Epoch can have multiple rows per match_key (the Phase 2 scoping pass
# found 6 duplicate Model values). Keep the most recent variant by
# Publication date.
epoch_subset = df_epoch[
    [
        "match_key",
        "Model",
        "Publication date",
        "Organization",
        "Training compute (FLOP)",
        "Parameters",
        "Training hardware",
        "Training power draw (W)",
    ]
].copy()
epoch_subset = epoch_subset.sort_values(
    "Publication date", ascending=False, na_position="last"
)
epoch_subset = epoch_subset.drop_duplicates("match_key", keep="first")
epoch_subset = epoch_subset.rename(columns={"Model": "epoch_model_name"})
print(f"epoch_subset (deduped on match_key): {epoch_subset.shape}")

epoch_subset (deduped on match_key): (3485, 8)


In [15]:
df_models = df_jegham.merge(epoch_subset, on="match_key", how="left").drop(
    columns="match_key"
)
print(f"df_models after left-merge: {df_models.shape}")
assert len(df_models) == len(df_jegham), (
    f"left-merge changed row count: {len(df_jegham)} -> {len(df_models)}"
)

df_models after left-merge: (194, 105)


In [16]:
validate_and_write(
    df_models,
    filename="models_enriched",
    must_have_columns=[
        "Model",
        "Query Length",
        "Mean Combined Energy (Wh)",
        "Mean Combined Carbon (gCO2e)",
        "Mean Combined Water (Site & Source, mL)",
        "epoch_model_name",
        "Publication date",
        "Organization",
        "Training compute (FLOP)",
        "Parameters",
        "Training hardware",
        "Training power draw (W)",
    ],
    expected_min_rows=194,
)

models_enriched: rows=194 cols=105


WindowsPath('../data/processed/integrated/models_enriched.parquet')

In [17]:
n_epoch_pop = df_models["epoch_model_name"].notna().sum()
n_epoch_null = df_models["epoch_model_name"].isna().sum()
print(f"models_enriched: rows with epoch_model_name populated: {n_epoch_pop}")
print(f"models_enriched: rows with epoch_model_name NULL:      {n_epoch_null}")
n_pw = df_models["Training power draw (W)"].notna().sum()
print(f"models_enriched: rows with Training power draw (W) populated: {n_pw}")

models_enriched: rows with epoch_model_name populated: 134
models_enriched: rows with epoch_model_name NULL:      60
models_enriched: rows with Training power draw (W) populated: 0


**Disposition for unmatched Jegham models.** The unmatched rows lack
training-side metadata in Epoch — typically because they post-date
Epoch's snapshot (e.g. the Claude 4 family) or because the hosted
variant naming pattern (`'GPT-4o (Aug)'`, `'DeepSeek R1 (Azure)'`)
does not collapse cleanly to an Epoch entry even after stripping
parens. Component 2 will display training-power-draw markers only for
the matched subset; unmatched rows are visually flagged. No silent
drops — the rows are present in `models_enriched.parquet` with NaN
Epoch columns.

**Disposition for `Training power draw (W)`.** Training power draw (W)
is included in the schema for completeness but is empty across all 134
matched rows. Epoch tracks this column primarily for older research-era
models (CNNs, BERT-class architectures) where labs published wall-clock
GPU power figures; for the hosted frontier LLMs Jegham covers, the labs
do not disclose training power draw and Epoch leaves the field NaN
rather than estimate it. Components 2 and 4 use Training compute (FLOP)
and Parameters as their primary training-side axes, which have 34% and
26% coverage respectively in the matched subset and are the brief's
planned axes regardless.

## E. Build `regions_geographic.parquet` (per-region context layer)

Per-region context table. Mixed grain: US rows at state level, other
rows at country level. Built from the AWS and Azure multiplier files
(regions + WUE + CIF), enriched with Ember grid carbon intensity,
Aqueduct water stress, and IM3 US facility footprint.

Feeds Component 3 (geographic map) directly; provides geographic
context for Components 5 and 7.

### E.1 Region geography mappings — controlled stop

The AWS and Microsoft multiplier files publish region geography as
human-readable strings (`'US East (Northern Virginia)'`, `'East US'`,
etc.) — no native ISO-3 country code or two-letter state postal code.
We need a small inline mapping `region_name → (iso3, state_abb)`.

This cell prints the unique region values from `df_aws['Region']` and
`df_ms['Region']` and asserts that every region is in the mapping.
The mapping starts with placeholder entries; the assertion will fail
as a controlled stop, listing every region that still needs an entry.
After the user fills the dictionaries in chat, this cell will pass
and the rest of Section E will run.

In [18]:
# AWS region → (ISO-3 country code, US state postal or None)
# Source: AWS Regional Services List, https://aws.amazon.com/about-aws/global-infrastructure/
aws_region_geography = {
    'Australia (Sydney)':           ('AUS', None),
    'Canada (Central) (Ontario)':   ('CAN', None),
    'Europe (Frankfurt)':           ('DEU', None),
    'Europe (Ireland)':             ('IRL', None),
    'Europe (London)':              ('GBR', None),
    'Middle East (UAE)':            ('ARE', None),
    'South America (São Paulo)':    ('BRA', None),
    'US East (Northern Virginia)':  ('USA', 'VA'),
    'US East (Ohio)':               ('USA', 'OH'),
    'US West (Oregon)':             ('USA', 'OR'),
}

# Microsoft Azure region → (ISO-3 country code, US state postal or None)
# Source: Azure geographies, https://azure.microsoft.com/en-us/explore/global-infrastructure/geographies/
# 'Average' is a synthetic aggregate row and is dropped in Section E.2.
ms_region_geography = {
    'Australia East':               ('AUS', None),
    'Brazil South':                 ('BRA', None),
    'Canada Central':               ('CAN', None),
    'Central India':                ('IND', None),
    'East Asia':                    ('HKG', None),
    'East US (PJM)':                ('USA', 'VA'),
    'East US 2 (PJM)':              ('USA', 'VA'),
    'France Central':               ('FRA', None),
    'Germany West Central':         ('DEU', None),
    'Italy North':                  ('ITA', None),
    'Japan East':                   ('JPN', None),
    'Korea Central':                ('KOR', None),
    'North Central US (MISO)':      ('USA', 'IL'),
    'North Europe':                 ('IRL', None),
    'Qatar Central':                ('QAT', None),
    'South Africa North':           ('ZAF', None),
    'South Central US (ERCOT)':     ('USA', 'TX'),
    'South East Asia':              ('SGP', None),
    'South India':                  ('IND', None),
    'Spain Central':                ('ESP', None),
    'Sweden Central':               ('SWE', None),
    'Switzerland North':            ('CHE', None),
    'UAE North':                    ('ARE', None),
    'UK South':                     ('GBR', None),
    'West Europe':                  ('NLD', None),
    'West US (CISO)':               ('USA', 'CA'),
    'West US 3 (APS)':              ('USA', 'AZ'),
}

aws_regions = sorted(df_aws["Region"].dropna().unique().tolist())
# Exclude 'Average' from the assertion — Microsoft includes this as a synthetic
# aggregate row, not a real region. It is dropped from df_ms in E.2.
ms_regions = sorted(
    r for r in df_ms["Region"].dropna().unique().tolist() if r != "Average"
)

print("AWS unique regions:")
for r in aws_regions:
    print(f"  {r!r}")
print(f"\nMicrosoft unique regions ({len(ms_regions)}):")
for r in ms_regions:
    print(f"  {r!r}")

aws_missing = [r for r in aws_regions if r not in aws_region_geography]
ms_missing = [r for r in ms_regions if r not in ms_region_geography]
print(
    f"\nAWS regions missing from mapping:        {len(aws_missing)} of {len(aws_regions)}"
)
print(
    f"Microsoft regions missing from mapping:  {len(ms_missing)} of {len(ms_regions)}"
)
assert not aws_missing, f"AWS regions need geography mapping: {aws_missing}"
assert not ms_missing, f"Microsoft regions need geography mapping: {ms_missing}"

AWS unique regions:
  'Australia (Sydney)'
  'Canada (Central) (Ontario)'
  'Europe (Frankfurt)'
  'Europe (Ireland)'
  'Europe (London)'
  'Middle East (UAE)'
  'South America (São Paulo)'
  'US East (Northern Virginia)'
  'US East (Ohio)'
  'US West (Oregon)'

Microsoft unique regions (27):
  'Australia East'
  'Brazil South'
  'Canada Central'
  'Central India'
  'East Asia'
  'East US (PJM)'
  'East US 2 (PJM)'
  'France Central'
  'Germany West Central'
  'Italy North'
  'Japan East'
  'Korea Central'
  'North Central US (MISO)'
  'North Europe'
  'Qatar Central'
  'South Africa North'
  'South Central US (ERCOT)'
  'South East Asia'
  'South India'
  'Spain Central'
  'Sweden Central'
  'Switzerland North'
  'UAE North'
  'UK South'
  'West Europe'
  'West US (CISO)'
  'West US 3 (APS)'

AWS regions missing from mapping:        0 of 10
Microsoft regions missing from mapping:  0 of 27


### E.2 Build the long-format `provider_region` table

Drop Microsoft's synthetic `'Average'` aggregate row, then concatenate
AWS and Azure rows into a long-format frame with columns
`[provider, region_name, iso3, state_abb, wue_site, cif_site,
wue_source, cif_source]`. Microsoft only ships single-block WUE/CIF,
so its `wue_source` / `cif_source` are NaN.

In [19]:
# Drop synthetic aggregate rows from both multiplier files. Microsoft tags
# its aggregate as Region == 'Average'. AWS leaves its aggregate's Region
# blank (NaN) — same intent, different convention.
df_aws = df_aws[df_aws["Region"].notna()].copy()
df_ms = df_ms[df_ms["Region"] != "Average"].copy()

# AWS columns after skiprows=1: 'WUE Source' / 'CIF' are the SITE values
# (the upstream header label is misleading); 'WUE Source.1' / 'CIF.1' are
# the source-level values produced by applying 'Multiplier'.
aws_long = pd.DataFrame(
    {
        "provider": "AWS",
        "region_name": df_aws["Region"].values,
        "wue_site": df_aws["WUE Source"].values,
        "cif_site": df_aws["CIF"].values,
        "wue_source": df_aws["WUE Source.1"].values,
        "cif_source": df_aws["CIF.1"].values,
    }
)
aws_long["iso3"] = aws_long["region_name"].map(lambda r: aws_region_geography[r][0])
aws_long["state_abb"] = aws_long["region_name"].map(
    lambda r: aws_region_geography[r][1]
)

# Microsoft has no site/source split — single WUE + CIF block.
ms_long = pd.DataFrame(
    {
        "provider": "Azure",
        "region_name": df_ms["Region"].values,
        "wue_site": df_ms["WUE Source"].values,
        "cif_site": df_ms["CIF"].values,
        "wue_source": np.nan,
        "cif_source": np.nan,
    }
)
ms_long["iso3"] = ms_long["region_name"].map(lambda r: ms_region_geography[r][0])
ms_long["state_abb"] = ms_long["region_name"].map(lambda r: ms_region_geography[r][1])

provider_region = pd.concat([aws_long, ms_long], ignore_index=True)[
    [
        "provider",
        "region_name",
        "iso3",
        "state_abb",
        "wue_site",
        "cif_site",
        "wue_source",
        "cif_source",
    ]
]
print(f"provider_region shape: {provider_region.shape}")
print(f"  AWS rows:   {(provider_region['provider'] == 'AWS').sum()}")
print(f"  Azure rows: {(provider_region['provider'] == 'Azure').sum()}")

provider_region shape: (37, 8)
  AWS rows:   10
  Azure rows: 27


### E.3 Enrich with Ember grid carbon intensity (latest year)

Look up grid carbon intensity by state postal for US rows and by
ISO-3 country code for non-US rows. We take the most recent year
available per region — the analytical comparisons in Phase 4 are
contemporaneous, not historical.

In [20]:
ember_us_lookup = (
    df_ember_us.sort_values("Year")
    .drop_duplicates("State code", keep="last")
    .set_index("State code")["Value"]
)
ember_yr_lookup = (
    df_ember_yr.sort_values("Year")
    .drop_duplicates("ISO 3 code", keep="last")
    .set_index("ISO 3 code")["Value"]
)


def lookup_grid_carbon(row):
    if pd.notna(row["state_abb"]):
        return ember_us_lookup.get(row["state_abb"], np.nan)
    return ember_yr_lookup.get(row["iso3"], np.nan)


provider_region["grid_carbon_intensity_gco2_kwh"] = provider_region.apply(
    lookup_grid_carbon, axis=1
)
print(
    f"grid_carbon_intensity populated: "
    f"{provider_region['grid_carbon_intensity_gco2_kwh'].notna().sum()} of "
    f"{len(provider_region)}"
)

grid_carbon_intensity populated: 37 of 37


### E.4 Enrich with Aqueduct baseline water stress (`bws`, `Tot`)

US rows look up `df_aq_us['score']` by state postal. Non-US rows look
up `df_aq_country['score']` by ISO-3 country code.

In [21]:
aq_us_lookup = df_aq_us.set_index("state_abb")["score"]
aq_country_lookup = df_aq_country.set_index("gid_0")["score"]


def lookup_water_stress(row):
    if pd.notna(row["state_abb"]):
        return aq_us_lookup.get(row["state_abb"], np.nan)
    return aq_country_lookup.get(row["iso3"], np.nan)


provider_region["water_stress_score"] = provider_region.apply(
    lookup_water_stress, axis=1
)
print(
    f"water_stress_score populated: "
    f"{provider_region['water_stress_score'].notna().sum()} of "
    f"{len(provider_region)}"
)

water_stress_score populated: 35 of 37


### E.5 Enrich US rows with IM3 facility footprint

Aggregate the campus-layer IM3 atlas to state level (count + total
sqft), then merge onto US rows by state postal. Non-US rows keep NaN
for both facility columns.

In [22]:
im3_state_agg = df_im3.groupby("state_abb").agg(
    us_facility_count_in_state=("id", "count"),
    us_facility_sqft_total_in_state=("sqft", "sum"),
)

provider_region = provider_region.merge(
    im3_state_agg, left_on="state_abb", right_index=True, how="left"
)

us_with_im3 = provider_region[
    provider_region["state_abb"].notna()
    & provider_region["us_facility_count_in_state"].notna()
]
print(f"US rows with IM3 facility data: {len(us_with_im3)}")

US rows with IM3 facility data: 9


In [23]:
validate_and_write(
    provider_region,
    filename="regions_geographic",
    must_have_columns=[
        "provider",
        "region_name",
        "iso3",
        "state_abb",
        "wue_site",
        "cif_site",
        "wue_source",
        "cif_source",
        "grid_carbon_intensity_gco2_kwh",
        "water_stress_score",
        "us_facility_count_in_state",
        "us_facility_sqft_total_in_state",
    ],
    expected_min_rows=37,
)

regions_geographic: rows=37 cols=12


WindowsPath('../data/processed/integrated/regions_geographic.parquet')

In [24]:
print(f"\nregions_geographic shape: {provider_region.shape}")
print(f"columns: {list(provider_region.columns)}")
print("\nhead(5):")
print(provider_region.head(5).to_string())

n_total = len(provider_region)
n_grid = provider_region["grid_carbon_intensity_gco2_kwh"].notna().sum()
n_water = provider_region["water_stress_score"].notna().sum()
n_us = provider_region["state_abb"].notna().sum()
n_us_im3 = (
    provider_region["state_abb"].notna()
    & provider_region["us_facility_count_in_state"].notna()
).sum()

print("\nJoin coverage:")
print(
    f"  grid_carbon_intensity: {n_grid} / {n_total}  "
    f"({100 * n_grid / n_total:.0f}%)"
)
print(
    f"  water_stress_score:    {n_water} / {n_total}  "
    f"({100 * n_water / n_total:.0f}%)"
)
print(
    f"  US rows with IM3 data: {n_us_im3} / {n_us}    "
    f"({100 * n_us_im3 / n_us:.0f}% of US)"
)

# Verification gate: flag any row where grid_carbon or water_stress
# is NaN. The user's brief sets a 95% threshold; flag the offenders.
for col, label in [
    ("grid_carbon_intensity_gco2_kwh", "grid_carbon_intensity"),
    ("water_stress_score", "water_stress_score"),
]:
    pct = 100 * provider_region[col].notna().sum() / n_total
    if pct < 95:
        bad = provider_region.loc[
            provider_region[col].isna(),
            ["provider", "region_name", "iso3", "state_abb"],
        ]
        print(f"\n  WARNING: {label} coverage {pct:.0f}% below 95% — failed regions:")
        print(bad.to_string(index=False))


regions_geographic shape: (37, 12)
columns: ['provider', 'region_name', 'iso3', 'state_abb', 'wue_site', 'cif_site', 'wue_source', 'cif_source', 'grid_carbon_intensity_gco2_kwh', 'water_stress_score', 'us_facility_count_in_state', 'us_facility_sqft_total_in_state']

head(5):
  provider                  region_name iso3 state_abb  wue_site  cif_site  wue_source  cif_source  grid_carbon_intensity_gco2_kwh  water_stress_score  us_facility_count_in_state  us_facility_sqft_total_in_state
0      AWS   Canada (Central) (Ontario)  CAN      None      8.18      0.11       24.54        0.33                          190.72            1.233315                         NaN                              NaN
1      AWS  US East (Northern Virginia)  USA        VA      2.27      0.39       13.62        2.34                          327.17            2.562449                        31.0                       56960153.0
2      AWS               US East (Ohio)  USA        OH      2.27      0.39        6.81 

## F. Final summary

Smoke test for the Phase 2 verification gate. Print, for each
integrated parquet, the source name (filename), filepath, row count,
column count, and the full column list.

In [25]:
print("=== Phase 2 outputs ===")
for parquet_path in sorted(INTEGRATED.glob("*.parquet")):
    df = pd.read_parquet(parquet_path)
    rel = parquet_path.relative_to(INTEGRATED.parent.parent)
    print(f"\n{parquet_path.name}")
    print(f"  filepath: {rel}")
    print(f"  rows:     {len(df)}")
    print(f"  cols:     {len(df.columns)}")
    print(f"  columns:  {list(df.columns)}")

=== Phase 2 outputs ===

models_enriched.parquet
  filepath: processed\integrated\models_enriched.parquet
  rows:     194
  cols:     105
  columns:  ['SnapshotDate', 'Model', 'ContextWindow', 'API ID', 'Artificial Analysis Intelligence Index', 'Terminal-Bench Hard (Agentic Coding & Terminal Use)', 'Tau2-Bench Telecom (Agentic Tool Use)', 'AA-LCR (Long Context Reasoning)', "Humanity's Last Exam (Reasoning & Knowledge)", 'MMLU-Pro (Reasoning & Knowledge)', 'GPQA Diamond (Scientific Reasoning)', 'LiveCodeBench (Coding)', 'SciCode (Coding)', 'IFBench (Instruction Following)', 'AIME 2025 (Competition Math)', 'MedianTokens/s', 'P5Tokens/s', 'P25Tokens/s', 'P75Tokens/s', 'P95Tokens/s', 'MedianFirst Chunk (s)', 'First AnswerToken (s)', 'P5First Chunk (s)', 'P25First Chunk (s)', 'P75First Chunk (s)', 'P95First Chunk (s)', 'length', 'Query Length', 'Hardware', 'Host', 'GPUs Power Draw', 'Non-GPUs Power Draw', 'Min GPU Power Utilization', 'Max GPU Power Utilization', 'Non-GPU Power Utilization',